# **Ejecución LLM**
En este noteebook hacemos que los distintos modelos de LLM evalúen las distintas imágenes de explicabilidad de los distintos métodos.

Para usar diferentes modelos de LLM, o imágenes asociadas a distintos modelos predictores basta modificar las variables `llm_model_name`, `architecture` y `dataset`

**Necesario archivo auxiliar prompts.py**

In [ ]:
# TODOS LOS MODELOS
#"lmstudio-community/Qwen3.5-2B-GGUF",
#"lmstudio-community/Qwen3.5-4B-GGUF",
#"lmstudio-community/Qwen3.5-9B-GGUF",
#"lmstudio-community/Qwen3.5-27B-GGUF",
#"lmstudio-community/Qwen3.8-27B-GGUF",

#"lmstudio-community/gemma-4-E2B-it-GGUF",
#"lmstudio-community/gemma-4-E4B-it-GGUF",
#"lmstudio-community/gemma-4-12b-it-GGUF",
#"lmstudio-community/gemma-4-26B-A4B-it-GGUF"

In [ ]:
llm_model_name =  "gemma-4-E2B-it-GGUF"

In [ ]:
# Arquitectura
# "dataset1_microsoft"
# "dataset2_microsoft"
# "dataset3_microsoft"
# "dataset1_googleViT"
# "dataset2_googleViT"
# "dataset3_googleViT"
# "dataset1_facebook_dinov2"
# "dataset2_facebook_dinov2"
# "dataset3_facebook_dinov2"
# "dataset3_facebook_dinov2_small_destilado"

In [ ]:
architecture = "dataset3_googleViT"

In [ ]:
# MÉTODOS XAI
# "attention_rollout"
# "expected_gradients"
# "gradcam"
# "integrated_gradients"
# "lime"
# "patch_occlusion"
# "pca"
# "rise"
# "scorecam"
# "shap"
# "smoothgrad"

# methods = ["attention_rollout",
#           "expected_gradients",
#           "gradcam",
#           "integrated_gradients",
#           "lime",
#           "patch_occlusion",
#           "pca",
#           "rise",
#           "scorecam",
#           "shap",
#           "smoothgrad"]
methods = ["attention_rollout","expected_gradients"]

In [ ]:
# method = "gradcam"

In [ ]:
# Datasets
# "PAD-UFES"
# "fitzpatrick_stratified_ninelabel_scale"
# "PH2"

In [ ]:
data_name = "PH2"

In [ ]:
# esta linea solo necesitas ejecutarla una vez para instalar lmstudio
!curl -fsSL https://lmstudio.ai/install.sh | bash

##################################################### 100.0%
Verifying checksum...
Installing llmster...
(node:1523) ExperimentalWarning: Single executable application is an experimental feature and might change at any time
(Use `llmster --trace-warnings ...` to show where the warning was created)
(node:1523) [DEP0040] DeprecationWarning: The `punycode` module is deprecated. Please use a userland alternative instead.
Install completed at /root/.lmstudio/llmster/0.0.21-2.
Installation finished successfully! llmster is ready to launch.
To start the daemon, run:

    lms daemon up

To add lms to your PATH, either restart your shell or run:
    export PATH="/root/.lmstudio/bin:$PATH"


In [ ]:
!pip install lmstudio>=1.5.0

In [ ]:
!~/.lmstudio/bin/lms daemon up

Waking up LM Studio service...
llmster started (PID: 1677).


In [ ]:
!mkdir -p ~/.lmstudio/models/lmstudio-community/{llm_model_name}

In [ ]:
# este bloque solo necesitas ejecutarlo una vez
!curl -LsSf https://astral.sh/uv/install.sh | sh
!mkdir -p ~/.temp-hf-cli
!cd ~/.temp-hf-cli && uv venv --python 3.12 && uv pip install --python ~/.temp-hf-cli/.venv/bin/python huggingface_hub

downloading uv 0.12.5 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using CPython 3.12.14
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Resolved 15 packages in 647ms
Prepared 15 packages in 519ms
Installed 15 packages in 43ms
 + anyio==4.14.2
 + certifi==2026.7.22
 + click==8.4.2
 + filelock==3.32.3
 + fsspec==2026.7.0
 + h11==0.16.0
 + hf-xet==1.6.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.28.0
 + idna==3.19
 + packaging==26.3
 + pyyaml==6.0.3
 + tqdm==4.70.0
 + typing-extensions==4.16.0


In [ ]:
!~/.temp-hf-cli/.venv/bin/hf sync --token mi_token \
    hf://buckets/z72pepee/explainability-selection/{llm_model_name} \
    ~/.lmstudio/models/lmstudio-community/{llm_model_name}

Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.
Scanning remote bucket (2 files)
Scanning local directory (0 files)
Comparing files (2 paths)
Sync plan: hf://buckets/z72pepee/explainability-selection/gemma-4-E2B-it-GGUF -> /root/.lmstudio/models/lmstudio-community/gemma-4-E2B-it-GGUF
  Uploads: 0
  Downloads: 2
  Deletes: 0
  Skips: 0
Syncing...


Sync completed.


In [ ]:
# dale a RESTART al notebook si este bloque da error

import lmstudio as lms

# ignoramos los timeouts y esperamos el tiempo que sea necesario
lms.set_sync_api_timeout(None)


model = lms.llm(
    f"lmstudio-community/{llm_model_name}",
    config={
        "contextLength": 32000,
        "gpu": {
            "ratio": 1.0  # 1.0 = 100% en GPU
        }
    }
)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("z72pepee/skin-gan-review-baseline", data_name, token="mi_token")

README.md:   0%|          | 0.00/35.4k [00:00<?, ?B/s]

PH2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 11.6MB            

PH2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

PH2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.44MB            

PH2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

PH2/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.49MB            

PH2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/160 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/20 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
import os
from huggingface_hub import HfApi
from huggingface_hub import batch_bucket_files
import io
from pathlib import Path
from huggingface_hub import list_bucket_tree
from huggingface_hub import download_bucket_files

os.environ["HF_TOKEN"] = "mi_token"

bucket_id = "z72pepee/tfm-explicabilidad"

token = os.environ.get("HF_TOKEN")
if not token:
    raise EnvironmentError("Define HF_TOKEN en el entorno antes de ejecutar esta celda")

api = HfApi(token=token)


for method in methods:
    carpetas_objetivo = [f"Representantes/{method}/", f"{architecture}/Resultados_{method.upper()}"]


    # Obtenemos la lista exacta de archivos dentro de esa carpeta
    archivos_remotos = [
        item.path
        for carpeta_objetivo in carpetas_objetivo
        for item in list_bucket_tree(bucket_id, prefix=carpeta_objetivo)
        if item.type == "file"
    ]
    #print(archivos_remotos[0])
    # Construimos la lista de tuplas que necesita download_bucket_files
    # Formato: (ruta_en_el_bucket, ruta_en_tu_disco_local)
    # Al poner (ruta, ruta) se guardarán conservando la misma estructura de carpetas.
    archivos_a_descargar = [
        (ruta, ruta)
        for ruta in archivos_remotos
    ]

    print(f"¡Se han encontrado {len(archivos_a_descargar)} archivos para descargar!")

    # Descargamos los archivos en bloque
    if archivos_a_descargar:
        download_bucket_files(
            bucket_id,
            files=archivos_a_descargar
        )
        print("¡Descarga completada con éxito!")
    else:
        print("No se encontró nada para descargar (la carpeta no existe o está vacía).")

¡Se han encontrado 30 archivos para descargar!


¡Descarga completada con éxito!
¡Se han encontrado 30 archivos para descargar!


¡Descarga completada con éxito!


In [ ]:
import re
from typing import List, Optional

def extract_evaluation_score(lines: List[str]) -> Optional[float]:
    """
    Extracts the evaluation score (1-5), always prioritizing the last occurrence in the text.
    It strictly accepts numbers that are placed immediately after a <result> tag.
    """
    full_text = " ".join(lines).strip()

    # re.findall searches for ALL matches in the text from start to finish.
    # It looks for the <result> tag, ignores whitespaces/random characters,
    # and captures the exact digit (1 to 5).
    matches = re.findall(r'<result>[^\d]*([1-5])', full_text, re.IGNORECASE)

    # If at least one valid tag was found...
    if matches:
        # [-1] grabs the LAST match in the list (representing the LLM's final decision).
        return float(matches[-1])

    # If the model completely ignored the formatting, return None to trigger a retry.
    return None

In [ ]:
# reintentos hasta lograr un valor numerico

import time
import os
import shutil
from pathlib import Path
from io import BytesIO
from PIL import Image as PILImage
from datasets import Image
from prompts import obtener_prompt
import pandas as pd

for method in methods:

  # Rutas y preparación (Igual que antes)
  carpeta_representantes = Path(f"Representantes/{method}/")
  carpeta_metodo = Path(f"{architecture}/Resultados_{method.upper()}")
  root_image = Path( "./ejemplos")

  root_image = Path("./ejemplos")
  if root_image.exists():
      shutil.rmtree(root_image)
  root_image.mkdir(parents=True, exist_ok=True)
  for img in carpeta_representantes.iterdir():
      if img.is_file():
          shutil.copy(img, root_image / img.name)

  carpeta_csv = Path(f"LLM_{architecture}/{llm_model_name}/{method}")
  carpeta_csv.mkdir(parents=True, exist_ok=True)

  # Configurar el dataset
  resultados_finales = []

  dataset['test'] = dataset['test'].cast_column("image", Image(decode=False))


  # El Bucle
  for example in dataset['test']:
      # Extraemos la ruta/nombre
      ruta_dataset = Path(example["image"]["path"])
      nombre_archivo = ruta_dataset.name # Coge solo "foto1.jpg", ignorando carpetas previas si las hay
      ruta_homonima = carpeta_metodo / nombre_archivo
      if ruta_homonima.exists():

          img_hf = PILImage.open(BytesIO(example["image"]["bytes"])).convert("RGB")

          # La guardamos como 6_o
          img_hf.save(root_image / "6_o.jpg")

          # Copiamos y renombramos la local a 6
          shutil.copy(ruta_homonima, root_image / "6.jpg")

          # --- AQUÍ EJECUTAS TU PROCESO CON LAS 12 IMÁGENES ---
          print(f"Lista para trabajar con el par de: {nombre_archivo}")

          #root_image = "./ejemplos"
          image_paths = os.listdir(root_image)
          image_paths.sort()
          image_paths = [os.path.join(root_image, i) for i in image_paths]
          print(image_paths)
          images = [lms.prepare_image(i) for i in image_paths]

          # def search_value(lines):
          #     lines.reverse()
          #     for i in lines:
          #         i = i.strip().split("<result>")
          #         try:
          #             value = float(i[1][0])
          #             return value
          #         except:
          #             pass
          #         try:
          #             value = float(i[1].strip().split()[0])
          #             return value
          #         except:
          #             pass

          for t in range(2):
              print("Intentando obtener un valor numerico...")

              # for gemma
              #chat = lms.Chat("<|think|>")

              chat = lms.Chat()

              prompt= obtener_prompt(method)

              # chat.add_user_message(
              #         """
              #         You are an expert assistant in evaluating explainability methods
              #         (XAI) applied to medical image classification models.
              #         Your task is to compare heatmaps generated by an explanation method
              #         with the actual location of a lesion in the original image, and
              #         assign a score from 1 to 5 indicating how well the area
              #         highlighted by the heatmap matches the lesion, where 1 is the worst match
              #         and 5 is the best match.

              #         Five pairs of images (10 images) are provided as examples, followed by 1 final pair to evaluate.
              #         The images are provided in pairs, with the first 2 images corresponding to a ranking score of 1, which is the worst result.
              #         The next, 3rd and 4th images represent ranking 2.
              #         This continues until images 9 and 10 represent ranking 5, which is the best result.
              #         """ +
              #         # Specific to this explanation model
              #         """
              #         The color red represents areas that the model takes strongly into account.
              #         Areas of the image with colors far from red are considered less important by the model.
              #         If the explainability method highlights the lesion area in red, or close to red, then it should receive a high score.

              #         The final 2 images are the pair you must evaluate.
              #         Assess whether the highlighted pixels correspond to the lesion in the image and respond only with the corresponding number.
              #         The color of the highlighted pixel does not need to match the color of the lesion.
              #         Be concise and direct.

              #         Image 11 contains a heatmap where red tones represent the area on which the model focused the most.
              #         Image 12 is the original lesion.
              #         """
              #         +
              #         """
              #         The goal is to assess whether the area highlighted by the explainability method overlaps with the area of the image where the lesion is located.
              #         Use the following template for the response, result tag plus blank space, followed by the score, blank space, and then the result tag again:
              #         \n <result> X <result>
              #         """,
              #         images=images
              #     )

              chat.add_user_message(prompt,images=images)

              start = time.time()
              prediction = model.respond(
                  chat,
                  config={
                      "temperature": 0.0,  # opcional
                  }
              )
              total_time = time.time() - start
              print(total_time)

              lines = prediction.content.splitlines()
              for i in lines:
                  print(i)

              value = extract_evaluation_score(lines)
              if value is not None:
                  break

          print(f"Valor final obtenido: {value}", total_time)
          # Añadimos los datos de esta iteración a nuestra lista
          resultados_finales.append({
              "imagen": nombre_archivo,
              "puntuacion": value,
              "tiempo_segundos": total_time
          })


  # Convertimos la lista a un DataFrame y lo guardamos
  df_resultados = pd.DataFrame(resultados_finales)
  df_resultados.to_csv(carpeta_csv / "puntuaciones_llm.csv", index=False)
  # GUARDADO LOCAL
  ruta_guardado_local = str(carpeta_csv / "puntuaciones_llm.csv")
  df_resultados.to_csv(ruta_guardado_local, index=False)

  print("\n¡Resultados guardados correctamente en CSV!")

  if not os.path.exists(ruta_guardado_local):
      raise FileNotFoundError(f"No se encuentra el fichero: {ruta_guardado_local}")

  # RUTA EN HUGGING FACE (Exactamente igual que en local)
  ruta_en_bucket = f"LLM_{architecture}/{llm_model_name}/{method}/puntuaciones_llm.csv"

  # SUBIDA AL BUCKET
  try:
      batch_bucket_files(
          bucket_id,
          add=[(ruta_guardado_local, ruta_en_bucket)]
      )
      print(f"Subida completada")
  except Exception as e:
      print(f" Error al subir al bucket HF: {e}")





Lista para trabajar con el par de: 06d1610f-aa93-4a71-af7e-d6b0e1760e45.png
['ejemplos/1.png', 'ejemplos/1_o.png', 'ejemplos/2.png', 'ejemplos/2_o.png', 'ejemplos/3.jpg', 'ejemplos/3_o.jpg', 'ejemplos/4.png', 'ejemplos/4_o.png', 'ejemplos/5.jpg', 'ejemplos/5_o.jpg', 'ejemplos/6.jpg', 'ejemplos/6_o.jpg']
Intentando obtener un valor numerico...
14.18888235092163
Thinking Process:

1.  **Analyze the Request and Role:** I am an expert assistant evaluating XAI heatmaps applied to medical image classification (specifically lesion location). The task is to compare a heatmap (Image 11) with the actual lesion location (Image 12) and assign a score from 1 (worst match) to 5 (best match).

2.  **Analyze the Calibration/Reference Examples (CRITICAL REFERENCE EXAMPLES):**
    *   Images 1 & 2 = Score 1 (Worst match).
    *   Images 3 & 4 = Score 2.
    *   Images 5 & 6 = Score 3.
    *   Images 7 & 8 = Score 4.
    *   Images 9 & 10 = Score 5 (Best match).
    *   Goal: Precision is key; noise/broa

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...lout/puntuaciones_llm.csv: 100%|##########| 1.31kB / 1.31kB            

Subida completada
Lista para trabajar con el par de: 06d1610f-aa93-4a71-af7e-d6b0e1760e45.png
['ejemplos/1.png', 'ejemplos/1_o.png', 'ejemplos/2.png', 'ejemplos/2_o.png', 'ejemplos/3.jpg', 'ejemplos/3_o.jpg', 'ejemplos/4.jpg', 'ejemplos/4_o.jpg', 'ejemplos/5.png', 'ejemplos/5_o.png', 'ejemplos/6.jpg', 'ejemplos/6_o.jpg']
Intentando obtener un valor numerico...
14.913742780685425
Thinking Process:

1.  **Analyze the Request and Role:** I am an expert assistant evaluating explainability methods (XAI) applied to medical image classification (specifically lesion detection). The task is to compare a heatmap (Image 11) with the actual lesion location (Image 12) and assign a score from 1 (worst match) to 5 (best match).

2.  **Analyze the Calibration/Reference Examples (Images 1-10):**
    *   1 & 2 = Score 1 (Worst match)
    *   3 & 4 = Score 2 (Poor)
    *   5 & 6 = Score 3 (Moderate)
    *   7 & 8 = Score 4 (Good)
    *   9 & 10 = Score 5 (Excellent)
    *   *Goal:* The evaluation must be

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ents/puntuaciones_llm.csv: 100%|##########| 1.31kB / 1.31kB            

Subida completada


In [ ]:
from huggingface_hub import batch_bucket_files
import os
import io
from pathlib import Path

carpeta = Path(f"LLM_{architecture}")
lista_tuplas = []

# rglob('*') busca recursivamente todos los elementos dentro de la carpeta y subcarpetas
for ruta_archivo in carpeta.rglob('*'):
    # Nos aseguramos de que es un archivo y no una carpeta
    if ruta_archivo.is_file():
        ruta_str = str(ruta_archivo)
        # Añadimos la tupla con la ruta repetida (ruta_local, ruta_destino)
        lista_tuplas.append((ruta_str, ruta_str))

print (lista_tuplas[0], len(lista_tuplas))

api.batch_bucket_files(
    bucket_id,
    add=lista_tuplas,
)

('LLM_dataset3_googleViT/gemma-4-E2B-it-GGUF/attention_rollout/puntuaciones_llm.csv', 'LLM_dataset3_googleViT/gemma-4-E2B-it-GGUF/attention_rollout/puntuaciones_llm.csv') 1


In [ ]:
!~/.lmstudio/bin/lms daemon down

In [ ]:
!rm -rf ~/.lmstudio/models/lmstudio-community/{llm_model_name}

In [ ]:
shutil.rmtree(f"./LLM_{architecture}/", ignore_errors=True)

In [ ]:
shutil.rmtree(f"./{architecture}/", ignore_errors=True)

In [ ]:
shutil.rmtree("./Representantes/", ignore_errors=True)